# Aplicações do Aprendizado Contrastivo

Um modelo de embeddings coloca fotos da mesma pessoa em pontos próximos. Com ele, reconhecer uma pessoa nova exige apenas cadastrar algumas fotos e comparar embeddings, sem retreinar a rede.

Este notebook monta um reconhecimento facial com modelos pré-treinados: as pessoas são cadastradas pela webcam e identificadas ao vivo. A parte da webcam precisa rodar na sua máquina, porque o OpenCV acessa a câmera local e abre uma janela própria para o vídeo.

In [ ]:
import os
import urllib.request
import zipfile

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import onnxruntime as ort
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from mediapipe.tasks.python import BaseOptions, vision
from sklearn.datasets import fetch_lfw_pairs, get_data_home
from sklearn.manifold import TSNE

## Os modelos

Usamos dois modelos. O detector de landmarks do MediaPipe encontra os rostos na imagem. O **ArcFace** do projeto InsightFace transforma cada rosto em um embedding. Os pesos do ArcFace são liberados apenas para pesquisa e uso não comercial.

O ArcFace vem no formato **ONNX**, um formato padrão para redes treinadas, executado pelo `onnxruntime`.

In [ ]:
def download(url, name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


face_detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=download("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task", "face.task")),
    num_faces=5,
))

if not os.path.exists("data/buffalo_sc/w600k_mbf.onnx"):
    with zipfile.ZipFile(download("https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_sc.zip", "buffalo_sc.zip")) as z:
        z.extractall("data/buffalo_sc")
options = ort.SessionOptions()
options.log_severity_level = 3   # esconde um aviso inofensivo sobre o tamanho do lote
arcface = ort.InferenceSession("data/buffalo_sc/w600k_mbf.onnx", options, providers=["CPUExecutionProvider"])

### Alinhamento

O ArcFace foi treinado com rostos de $112 \times 112$ pixels, com os olhos, o nariz e os cantos da boca sempre nas mesmas posições. A `find_faces` leva esses cinco pontos do MediaPipe às posições padrão e devolve os rostos alinhados e a caixa de cada um. A `embed` devolve os embeddings com norma 1.

In [ ]:
FIVE_POINTS = [468, 473, 1, 61, 291]   # íris, íris, nariz, canto da boca, canto da boca
TEMPLATE = np.array([[38.29, 51.70], [73.53, 51.50], [56.03, 71.74], [41.55, 92.37], [70.73, 92.20]], dtype=np.float32)


def find_faces(image):
    # image: RGB uint8 [H, W, 3]
    H, W = image.shape[:2]
    faces, boxes = [], []
    for landmarks in face_detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=image)).face_landmarks:
        points = np.array([[point.x * W, point.y * H] for point in landmarks], dtype=np.float32)   # [478, 2]
        M, _ = cv2.estimateAffinePartial2D(points[FIVE_POINTS], TEMPLATE)
        faces.append(cv2.warpAffine(image, M, (112, 112)))                  # [112, 112, 3]
        boxes.append(np.r_[points.min(axis=0), points.max(axis=0)].astype(int))   # x0, y0, x1, y1
    return faces, boxes


def embed(faces):
    x = (np.stack(faces).astype(np.float32) - 127.5) / 127.5   # [N, 112, 112, 3], entre -1 e 1
    z = arcface.run(None, {"input.1": x.transpose(0, 3, 1, 2)})[0]   # [N, 512]
    return z / np.linalg.norm(z, axis=1, keepdims=True)

## Embedding de um rosto

As fotos vêm do **LFW** (*Labeled Faces in the Wild*), um dataset de fotos de celebridades que o `scikit-learn` baixa, cerca de 230 MB. Algumas fotos têm mais de uma pessoa, então recortamos o centro da imagem, onde fica o rosto de interesse.

Pegamos três rostos: dois da mesma pessoa e um de outra. Cada rosto vira um vetor de 512 números.

In [ ]:
CENTER = slice(50, 200), slice(50, 200)   # recorte de 150 x 150 no centro da foto de 250 x 250

lfw = fetch_lfw_pairs(subset="test", color=True, resize=1.0, slice_=CENTER)   # [1000, 2, 150, 150, 3], entre 0 e 1
lfw_dir = os.path.join(get_data_home(), "lfw_home", "lfw_funneled")


def read_photo(name, number):
    image = cv2.imread(os.path.join(lfw_dir, name, f"{name}_{number:04d}.jpg"))
    return cv2.cvtColor(image[CENTER], cv2.COLOR_BGR2RGB)


examples = [("Arnold_Schwarzenegger", 1), ("Arnold_Schwarzenegger", 2), ("Vladimir_Putin", 1)]
faces = [find_faces(read_photo(name, number))[0][0] for name, number in examples]
z = embed(faces)   # [3, 512]
print(z.shape)
print(z[0, :5])

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, face, (name, _) in zip(axes, faces, examples):
    ax.imshow(face)
    ax.set_title(name)
    ax.axis("off")
plt.show()

Com norma 1, a similaridade do cosseno entre dois embeddings é o produto escalar deles. O produto `z @ z.T` calcula a similaridade de cada rosto com todos os outros.

In [ ]:
print(np.round(z @ z.T, 2))

A diagonal vale 1, porque cada rosto é igual a si mesmo. Os dois rostos da mesma pessoa têm similaridade alta, e os rostos de pessoas diferentes ficam perto de 0.

### Escolha do limiar

Duas fotos são da mesma pessoa se a similaridade passa de um limiar. Para escolhê-lo, usamos os mil pares de fotos do LFW: metade da mesma pessoa e metade de pessoas diferentes.

In [ ]:
pairs = (lfw.pairs * 255).astype(np.uint8)

similarities, same = [], []
for (image_a, image_b), target in zip(pairs, lfw.target):
    faces_a, _ = find_faces(image_a)
    faces_b, _ = find_faces(image_b)
    if faces_a and faces_b:
        z = embed([faces_a[0], faces_b[0]])
        similarities.append(z[0] @ z[1])
        same.append(target == 1)
similarities, same = np.array(similarities), np.array(same)

thresholds = np.linspace(-1, 1, 201)
accuracies = [((similarities > t) == same).mean() for t in thresholds]
face_threshold = thresholds[np.argmax(accuracies)]
print(f"{len(same)} pares, limiar {face_threshold:.2f}, acurácia {max(accuracies):.1%}")

plt.hist(similarities[same], bins=40, alpha=0.6, label="mesma pessoa")
plt.hist(similarities[~same], bins=40, alpha=0.6, label="pessoas diferentes")
plt.axvline(face_threshold, color="black", linestyle="--", label="limiar")
plt.xlabel("similaridade do cosseno")
plt.legend()
plt.show()

## O banco de rostos

O banco guarda as fotos alinhadas de cada pessoa, uma pasta por pessoa em `data/faces`. Para ele não começar vazio, cadastramos dez fotos de seis celebridades do LFW.

In [ ]:
FACES_DIR = "data/faces"
CELEBRITIES = ["Angelina_Jolie", "Arnold_Schwarzenegger", "Jennifer_Lopez",
               "Keanu_Reeves", "Serena_Williams", "Vladimir_Putin"]

for name in CELEBRITIES:
    folder = os.path.join(FACES_DIR, name)
    if os.path.exists(folder):
        continue
    os.makedirs(folder)
    for number in range(1, 11):
        faces, _ = find_faces(read_photo(name, number))
        if faces:
            cv2.imwrite(os.path.join(folder, f"{number:03d}.png"), cv2.cvtColor(faces[0], cv2.COLOR_RGB2BGR))


def load_database():
    labels, faces = [], []
    for name in sorted(os.listdir(FACES_DIR)):
        for file in sorted(os.listdir(os.path.join(FACES_DIR, name))):
            faces.append(cv2.cvtColor(cv2.imread(os.path.join(FACES_DIR, name, file)), cv2.COLOR_BGR2RGB))
            labels.append(name)
    return np.array(labels), faces, embed(faces)   # [N], N x [112, 112, 3], [N, 512]


labels, faces, embeddings = load_database()
for name in np.unique(labels):
    print(name, (labels == name).sum())

### Visualização dos embeddings

O **t-SNE** projeta os embeddings em 2D, mantendo perto os pontos que estão perto no espaço original. As distâncias entre os grupos no gráfico não têm significado; o que importa é quem fica junto de quem.

In [ ]:
def plot_embeddings():
    labels, faces, embeddings = load_database()
    colors = {name: plt.cm.tab10(k % 10) for k, name in enumerate(np.unique(labels))}
    points = TSNE(metric="cosine", random_state=0).fit_transform(embeddings)   # [N, 2]

    fig, ax = plt.subplots(figsize=(12, 10))
    for name, color in colors.items():
        ax.scatter(*points[labels == name].T, color=color, label=name)
    for face, point, name in zip(faces, points, labels):
        ax.add_artist(AnnotationBbox(OffsetImage(face, zoom=0.3), point, bboxprops={"edgecolor": colors[name], "linewidth": 3}))
    ax.legend()
    ax.axis("off")
    plt.show()


plot_embeddings()

As fotos de cada pessoa formam um grupo separado, mesmo com poses, idades e iluminações diferentes. A rede nunca foi treinada para separar essas seis pessoas.

## Cadastro pela webcam

A `run_webcam` lê a câmera e chama o `process` em cada frame, com a tecla apertada. A `capture` adiciona fotos de uma pessoa ao banco: aperte **espaço** para tirar uma foto, com um único rosto na imagem, e **q** para terminar. Tire de 5 a 10 fotos girando um pouco a cabeça e mudando a expressão.

In [ ]:
def run_webcam(process, source=0):
    camera = cv2.VideoCapture(source)   # 0 é a webcam; também aceita o caminho de um vídeo
    key = -1
    while True:
        ok, frame = camera.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)   # espelha na horizontal
        process(frame, key)
        cv2.imshow("webcam", frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):   # q ou Esc
            break
    camera.release()
    cv2.destroyAllWindows()


def draw_box(frame, box, text, color):
    x0, y0, x1, y1 = box
    cv2.rectangle(frame, (x0, y0), (x1, y1), color, 2)
    cv2.putText(frame, text, (x0, y0 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

In [ ]:
def capture(name):
    folder = os.path.join(FACES_DIR, name)
    os.makedirs(folder, exist_ok=True)

    def process(frame, key):
        faces, boxes = find_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))   # a webcam entrega BGR
        if key == ord(" ") and len(faces) == 1:
            path = os.path.join(folder, f"{len(os.listdir(folder)):03d}.png")
            cv2.imwrite(path, cv2.cvtColor(faces[0], cv2.COLOR_RGB2BGR))
        for box in boxes:
            draw_box(frame, box, f"{name}: {len(os.listdir(folder))} fotos", (0, 200, 255))

    run_webcam(process)


capture("Silvan")   # troque pelo seu nome; rode de novo com outro nome para cadastrar outra pessoa
plot_embeddings()

No gráfico, as suas fotos devem formar um grupo novo, longe das celebridades.

## Reconhecimento

Cada rosto da câmera é comparado com todas as fotos do banco. O rosto recebe o nome da foto mais similar, se a similaridade passar do limiar; senão, é um desconhecido.

Se você aparece como desconhecido, diminua o `face_threshold`. Se pessoas diferentes recebem o mesmo nome, aumente.

In [ ]:
labels, _, embeddings = load_database()


def recognize(frame, key):
    faces, boxes = find_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not faces:
        return
    similarities = embed(faces) @ embeddings.T   # [rostos, fotos do banco]
    for box, row in zip(boxes, similarities):
        best = row.argmax()
        if row[best] > face_threshold:
            draw_box(frame, box, f"{labels[best]} {row[best]:.2f}", (0, 200, 0))
        else:
            draw_box(frame, box, f"desconhecido {row[best]:.2f}", (0, 0, 230))


run_webcam(recognize)

Cadastrar alguém é calcular embeddings de algumas fotos, e a rede não muda.

Uma foto impressa ou um vídeo no celular também tem o rosto da pessoa, e o modelo os reconhece do mesmo jeito. Sistemas reais acrescentam uma **detecção de vivacidade** (*liveness*), que verifica se o rosto é de uma pessoa presente, por exemplo pedindo para piscar ou usando uma câmera de profundidade.